In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Importing important libraries

In [54]:
import pandas as pd
import numpy as np

# Loading the dataset

In [55]:
df=pd.read_csv('/kaggle/input/datasets/shubhamgusain101/ufo-dataset/ufo_sighting_data.csv',low_memory=False)

# Inspecting the table

In [56]:
df.head()

,Date_time,city,state/province,country,UFO_shape,length_of_encounter_seconds,described_duration_of_encounter,description,date_documented,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80332 entries, 0 to 80331
Data columns (total 11 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Date_time                        80332 non-null  object 
 1   city                             80332 non-null  object 
 2   state/province                   74535 non-null  object 
 3   country                          70662 non-null  object 
 4   UFO_shape                        78400 non-null  object 
 5   length_of_encounter_seconds      80332 non-null  object 
 6   described_duration_of_encounter  80332 non-null  object 
 7   description                      80317 non-null  object 
 8   date_documented                  80332 non-null  object 
 9   latitude                         80332 non-null  object 
 10  longitude                        80332 non-null  float64
dtypes: float64(1), object(10)
memory usage: 6.7+ MB


#### The dataset has maxium row count of 80332 with 10 columns in total

In [58]:
df.isna().sum()

Date_time                             0
city                                  0
state/province                     5797
country                            9670
UFO_shape                          1932
length_of_encounter_seconds           0
described_duration_of_encounter       0
description                          15
date_documented                       0
latitude                              0
longitude                             0
dtype: int64

#### Three of the columns have significant null values that need to be handled before using the dataset

# Cleaning the dataset

In [59]:
df.head()

,Date_time,city,state/province,country,UFO_shape,length_of_encounter_seconds,described_duration_of_encounter,description,date_documented,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611


# Transforming the dataset

## 1. Fix datetime Column (24:00 Fix & Type Conversion)
### Replaces 24:00 with 00:00 and converts the column to a proper datetime64 object.

In [62]:

# Replace '24:00' with '00:00' to handle midnight timestamps
df['Date_time'] = df['Date_time'].astype(str).str.replace(' 24:00', ' 00:00', regex=False)

# Convert to datetime format
df['Date_time'] = pd.to_datetime(df['Date_time'], format='%m/%d/%Y %H:%M', errors='coerce')

## 2. Extract Slicing Columns from Date_time
### Creates individual date/time features for Power BI filters and axes.

In [63]:
df['Year'] = df['Date_time'].dt.year
df['Month_Name'] = df['Date_time'].dt.month_name()
df['Day_Name'] = df['Date_time'].dt.day_name()
df['Hour'] = df['Date_time'].dt.hour

## 3. Clean latitude & longitude
### Strips out trailing non-numeric characters from latitude and converts both to floating point numbers.

In [64]:
# Extract valid decimal number format from latitude
df['latitude'] = df['latitude'].astype(str).str.extract(r'(-?\d+\.?\d*)')

# Convert both coordinates to float
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

## 4. Clean length_of_encounter_seconds
### Converts the duration to numeric float and filters out non-sensical non-positive values.

In [65]:
# Convert to float
df['length_of_encounter_seconds'] = pd.to_numeric(df['length_of_encounter_seconds'], errors='coerce')

# Set zero or negative durations to NaN
df.loc[df['length_of_encounter_seconds'] <= 0, 'length_of_encounter_seconds'] = np.nan

## 6. Clean country & state/province
### Converts region codes to uppercase and fills missing values with 'Unknown'.

In [66]:
# Uppercase and fill missing/blank values
df['country'] = df['country'].fillna('Unknown').astype(str).str.upper().replace({'': 'Unknown', 'NAN': 'Unknown'})
df['state/province'] = df['state/province'].fillna('Unknown').astype(str).str.upper().replace({'': 'Unknown', 'NAN': 'Unknown'})

## 6. Standardize UFO_shape
### Trims whitespace, capitalizes shape titles, and handles missing/unknown categories.

In [67]:
# Clean whitespace and apply title-case
df['UFO_shape'] = df['UFO_shape'].astype(str).str.strip().str.title()

# Replace missing, empty, or 'Unknown' values with 'Unspecified'
df['UFO_shape'] = df['UFO_shape'].replace(['Nan', 'None', '', 'Unknown'], 'Unspecified')

# Handling misisng values from country and state/province column
#### since many rows missing country have either state/province or the city column filled with infomation and vice versa logically thee columns can be filled with the information provided in the other columns

In [68]:
ca_provinces = ['ON', 'BC', 'QC', 'AB', 'MB', 'NS', 'NB', 'SK', 'NL', 'PE', 'NT', 'YT', 'NU']

# 1. Clean and uppercase both columns first
df['country'] = df['country'].fillna('UNKNOWN').replace(['', 'NAN', 'nan', 'Nan', 'NULL'], 'UNKNOWN').astype(str).str.strip().str.upper()
df['state/province'] = df['state/province'].fillna('UNKNOWN').replace(['', 'NAN', 'nan', 'Nan', 'NULL'], 'UNKNOWN').astype(str).str.strip().str.upper()

# 2. Smart fill country if state/province is known
# If state is known but country is UNKNOWN, infer US (or CA if in ca_provinces)
is_known_state = df['state/province'] != 'UNKNOWN'
is_unknown_country = df['country'] == 'UNKNOWN'

df.loc[is_known_state & is_unknown_country & df['state/province'].isin(ca_provinces), 'country'] = 'CA'
df.loc[is_known_state & is_unknown_country & ~df['state/province'].isin(ca_provinces), 'country'] = 'US'

In [69]:
df.head()

,Date_time,city,state/province,country,UFO_shape,length_of_encounter_seconds,described_duration_of_encounter,description,date_documented,latitude,longitude,Year,Month_Name,Day_Name,Hour
0,1949-10-10 20:30:00,san marcos,TX,US,Cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949,October,Monday,20
1,1949-10-10 21:00:00,lackland afb,TX,US,Light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082,1949,October,Monday,21
2,1955-10-10 17:00:00,chester (uk/england),UNKNOWN,GB,Circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667,1955,October,Monday,17
3,1956-10-10 21:00:00,edna,TX,US,Circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956,October,Wednesday,21
4,1960-10-10 20:00:00,kaneohe,HI,US,Light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960,October,Monday,20


In [70]:
df.isna().sum()

Date_time                           0
city                                0
state/province                      0
country                             0
UFO_shape                           0
length_of_encounter_seconds         3
described_duration_of_encounter     0
description                        15
date_documented                     0
latitude                            0
longitude                           0
Year                                0
Month_Name                          0
Day_Name                            0
Hour                                0
dtype: int64

### Now there are minimum misisng values

# Cleaning the dataset
### there are several columns in this dataset that are not neccessary for our evaluation or redundant so we will drop these columns from the dataset

In [71]:
# Define the list of columns to drop
columns_to_drop = [
    'city', 
    'described_duration_of_encounter', 
    'description', 
    'date_documented'
]

# Drop the specified columns safely
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [72]:
df.head()

,Date_time,state/province,country,UFO_shape,length_of_encounter_seconds,latitude,longitude,Year,Month_Name,Day_Name,Hour
0,1949-10-10 20:30:00,TX,US,Cylinder,2700.0,29.883056,-97.941111,1949,October,Monday,20
1,1949-10-10 21:00:00,TX,US,Light,7200.0,29.384210,-98.581082,1949,October,Monday,21
2,1955-10-10 17:00:00,UNKNOWN,GB,Circle,20.0,53.200000,-2.916667,1955,October,Monday,17
3,1956-10-10 21:00:00,TX,US,Circle,20.0,28.978333,-96.645833,1956,October,Wednesday,21
4,1960-10-10 20:00:00,HI,US,Light,900.0,21.418056,-157.803611,1960,October,Monday,20


# Saving the dataset 

In [74]:
df = df.drop_duplicates()
df.to_csv('ufo_sighting_data_cleaned.csv', index=False)